# Audio Compression & Spectrogram Analysis: Lossless (FLAC) vs. Lossy (MP3 Bitrates)

This notebook provides an interactive pipeline to:
1. **Upload or select an audio file** (`.flac`, `.wav`, `.mp3`, `.ogg`, `.m4a`) from your local machine (or generate a full-spectrum reference `.flac` audio file for testing).
2. **Convert lossless FLAC audio to MP3** across multiple bitrates (`320k`, `256k`, `192k`, `128k`, `64k`, `32k`) and optional formats (`ogg`, `wav`).
3. **Visualize and compare spectrograms** with and without compression:
   - **Full-Bandwidth Linear STFT Spectrograms** (to observe high-frequency brickwall cutoffs introduced by MP3 encoders).
   - **Residual / Difference Spectrograms** (to isolate exact time-frequency regions discarded by psychoacoustic masking).
   - **Mel-Frequency Spectrograms** (perceptually weighted frequency representation).
   - **Power Spectral Density (PSD) & Compression Trade-off Metrics** (file size reduction, compression ratio, Log-Spectral Distance, and estimated frequency roll-off).

## 1. Environment Setup & Library Imports

Ensure the notebook kernel is set to the project virtual environment (`.venv`) and that `ffmpeg` is installed on your system.

In [ ]:
import os
import shutil
import subprocess
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import IPython.display as ipd

# Check required third-party audio packages
missing_pkgs = []
for pkg in ["librosa", "soundfile", "ipywidgets"]:
    try:
        __import__(pkg)
    except ImportError:
        missing_pkgs.append(pkg)

if missing_pkgs:
    print(f"WARNING: Missing Python packages: {', '.join(missing_pkgs)}")
    print(f"Install them in .venv with: .venv/bin/pip install {' '.join(missing_pkgs)}")
else:
    import librosa
    import librosa.display
    import soundfile as sf
    import ipywidgets as widgets
    print("All required Python audio libraries loaded successfully.")

# Verify ffmpeg availability (required for MP3 encoding/decoding)
ffmpeg_path = shutil.which("ffmpeg")
if ffmpeg_path:
    print(f"ffmpeg detected at: {ffmpeg_path}")
else:
    print("WARNING: 'ffmpeg' was not found on PATH. MP3 conversion requires ffmpeg.")
    print("  Ubuntu/Debian : sudo apt-get install ffmpeg")
    print("  macOS         : brew install ffmpeg")
    print("  Conda         : conda install -c conda-forge ffmpeg")

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = False

WORKSPACE_DIR = Path("audio_workspace")
WORKSPACE_DIR.mkdir(exist_ok=True)
print(f"Working directory for audio files: {WORKSPACE_DIR.resolve()}")

## 2. Upload or Select Audio File from Local Machine

You can load a `.flac` (or `.wav`, `.mp3`, `.ogg`, `.m4a`) file from your computer using **any of the three methods below**:
- **Method A (Interactive Widget)**: Click **Upload Audio** below to select a file from your local machine.
- **Method B (Local File Path)**: Enter the path to a local audio file on your machine in the text box (or set `LOCAL_AUDIO_FILE`) and click **Load Local Path**.
- **Method C (Reference FLAC Generator)**: Click **Generate Reference FLAC** (also used as the default initial file) to synthesize a 44.1 kHz lossless `.flac` test track containing harmonic chords, a logarithmic frequency sweep ($50\text{ Hz} \to 21.5\text{ kHz}$), and broadband high-frequency air.

In [ ]:
# Global state holding the active lossless FLAC file path for analysis
ACTIVE_FLAC_PATH = None


def generate_reference_flac(output_path: Path = WORKSPACE_DIR / "reference_lossless.flac", sr: int = 44100, duration: float = 6.0) -> Path:
    """
    Generates a full-bandwidth (0 - 22.05 kHz) lossless FLAC audio file combining:
    - Musical harmonic series (rich midrange & high overtones)
    - Exponential chirp sweep (50 Hz -> 21,500 Hz) to make high-frequency cutoffs immediately visible
    - Broadband texture across all time frames
    """
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    audio = np.zeros_like(t)

    # 1. Harmonic chord progression with rich overtones up to 18 kHz
    fundamental_freqs = [220.0, 277.18, 329.63, 440.0]
    for f0 in fundamental_freqs:
        for harmonic in range(1, int((sr / 2 - 500) // f0)):
            amp = (1.0 / (harmonic ** 1.15)) * np.exp(-0.25 * (t % 2.0))
            audio += amp * np.sin(2 * np.pi * f0 * harmonic * t)

    # 2. Full-spectrum logarithmic frequency sweep (50 Hz to 21.5 kHz)
    sweep = signal.chirp(t, f0=50, f1=21500, t1=duration, method="logarithmic") * 0.35

    # 3. Broadband background texture so high-frequency shelf cutoffs are clearly visible across all frames
    rng = np.random.default_rng(42)
    broadband = rng.normal(0, 0.08, size=t.shape)

    # Combine and normalize to -1 dBFS
    combined = audio + sweep + broadband
    combined = combined / (np.max(np.abs(combined)) + 1e-8) * 0.89

    sf.write(str(output_path), combined, sr, format="FLAC", subtype="PCM_16")
    return output_path


def ensure_flac_source(input_path: Path) -> Path:
    """
    Ensures we have a baseline .flac file in WORKSPACE_DIR.
    If the user uploads a .flac file, it is used directly.
    If the user uploads another format (.wav, .mp3, .ogg), we also save a baseline .flac copy.
    """
    input_path = Path(input_path).expanduser().resolve()
    if not input_path.exists():
        raise FileNotFoundError(f"File not found: {input_path}")

    if input_path.suffix.lower() == ".flac":
        target_flac = WORKSPACE_DIR / input_path.name
        if input_path != target_flac.resolve():
            shutil.copy2(input_path, target_flac)
        return target_flac
    else:
        target_flac = WORKSPACE_DIR / f"{input_path.stem}_converted_source.flac"
        y, sr = librosa.load(str(input_path), sr=None, mono=False)
        if y.ndim > 1:
            y = y.T
        sf.write(str(target_flac), y, sr, format="FLAC")
        print(f"Converted uploaded '{input_path.name}' to FLAC baseline: {target_flac.name}")
        return target_flac


# --- Interactive Upload & File Selection UI ---
upload_widget = widgets.FileUpload(
    accept=".flac,.wav,.mp3,.ogg,.m4a,.aac",
    multiple=False,
    description="Upload Audio",
    button_style="primary",
)

path_input = widgets.Text(
    value="",
    placeholder="/path/to/your/local/audio.flac",
    description="Local Path:",
    layout=widgets.Layout(width="450px"),
)
load_path_btn = widgets.Button(description="Load Local Path", button_style="info")
demo_btn = widgets.Button(description="Generate Reference FLAC", button_style="success")
status_out = widgets.Output()


def on_upload_change(change):
    global ACTIVE_FLAC_PATH
    with status_out:
        status_out.clear_output()
        val = upload_widget.value
        if not val:
            return
        # Handle both ipywidgets v7 (dict) and v8 (tuple of dicts)
        file_info = list(val.values())[0] if isinstance(val, dict) else val[0]
        fname = file_info["name"]
        content = file_info["content"]
        raw_bytes = content.tobytes() if hasattr(content, "tobytes") else bytes(content)
        saved_path = WORKSPACE_DIR / fname
        with open(saved_path, "wb") as f:
            f.write(raw_bytes)
        ACTIVE_FLAC_PATH = ensure_flac_source(saved_path)
        size_kb = ACTIVE_FLAC_PATH.stat().st_size / 1024
        print(f"Uploaded and set active FLAC file: {ACTIVE_FLAC_PATH} ({size_kb:.1f} KB)")


def on_load_path_clicked(_):
    global ACTIVE_FLAC_PATH
    with status_out:
        status_out.clear_output()
        raw_path = path_input.value.strip()
        if not raw_path:
            print("Please enter a valid file path in the 'Local Path' field.")
            return
        try:
            ACTIVE_FLAC_PATH = ensure_flac_source(Path(raw_path))
            size_kb = ACTIVE_FLAC_PATH.stat().st_size / 1024
            print(f"Loaded local audio file as active FLAC: {ACTIVE_FLAC_PATH} ({size_kb:.1f} KB)")
        except Exception as exc:
            print(f"Error loading path: {exc}")


def on_demo_clicked(_):
    global ACTIVE_FLAC_PATH
    with status_out:
        status_out.clear_output()
        ACTIVE_FLAC_PATH = generate_reference_flac()
        size_kb = ACTIVE_FLAC_PATH.stat().st_size / 1024
        print(f"Generated reference lossless FLAC file: {ACTIVE_FLAC_PATH} ({size_kb:.1f} KB)")


upload_widget.observe(on_upload_change, names="value")
load_path_btn.on_click(on_load_path_clicked)
demo_btn.on_click(on_demo_clicked)

# Display controls
ui_row1 = widgets.HBox([upload_widget, demo_btn])
ui_row2 = widgets.HBox([path_input, load_path_btn])
ipd.display(widgets.VBox([ui_row1, ui_row2, status_out]))

# Optional direct path override: set LOCAL_AUDIO_FILE below if you prefer not to use widgets
LOCAL_AUDIO_FILE = None  # e.g., "/home/user/Music/sample.flac"

if LOCAL_AUDIO_FILE:
    ACTIVE_FLAC_PATH = ensure_flac_source(Path(LOCAL_AUDIO_FILE))
    print(f"Using LOCAL_AUDIO_FILE: {ACTIVE_FLAC_PATH}")
elif ACTIVE_FLAC_PATH is None:
    # Initialize with a reference FLAC file so subsequent cells can be run immediately
    ACTIVE_FLAC_PATH = generate_reference_flac()
    print(f"Default active file initialized to: {ACTIVE_FLAC_PATH.name} (Upload your own file above at any time!)")

## 3. Convert Audio from FLAC to MP3 (Multiple Bitrates) & Other Formats

The functions below convert a source `.flac` file into `.mp3` files at any target bitrate (e.g., `320k`, `256k`, `192k`, `128k`, `64k`, `32k`) as well as other formats (such as `ogg` or uncompressed `wav`) using `ffmpeg` (compatible with Python 3.10–3.14+):

- `convert_flac_to_mp3(input_flac_path, output_mp3_path, bitrate='192k')`: Converts a single FLAC file to MP3 at a specified bitrate.
- `convert_flac_to_multiple_bitrates(input_flac_path, bitrates, extra_formats)`: Batch-converts the source FLAC across a list of MP3 bitrates and returns metadata (file size, compression ratio, space saved).

In [ ]:
def convert_audio_ffmpeg(
    input_path: str | Path,
    output_path: str | Path,
    bitrate: str | None = "192k",
    codec: str | None = None,
) -> Path:
    """
    Convert an audio file using ffmpeg with explicit bitrate control.
    Works cleanly across all Python versions (including Python 3.13/3.14 where stdlib audioop was removed).
    """
    input_path = Path(input_path)
    output_path = Path(output_path)
    if not shutil.which("ffmpeg"):
        raise RuntimeError("ffmpeg is not installed or not on PATH. Please install ffmpeg to convert to MP3.")

    cmd = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(input_path)]
    if codec:
        cmd.extend(["-c:a", codec])
    if bitrate:
        cmd.extend(["-b:a", bitrate])
    cmd.append(str(output_path))
    subprocess.run(cmd, check=True)
    return output_path


def convert_flac_to_mp3(
    input_flac_path: str | Path,
    output_mp3_path: str | Path | None = None,
    bitrate: str = "192k",
) -> Path:
    """
    Convert a FLAC (or any supported audio) file to an MP3 file at the specified bitrate.

    Parameters
    ----------
    input_flac_path : str or Path
        Path to the source .flac audio file.
    output_mp3_path : str, Path, or None
        Destination path for the converted .mp3 file. If None, saves inside WORKSPACE_DIR.
    bitrate : str
        Target MP3 bitrate (e.g., '320k', '256k', '192k', '128k', '64k', '32k').

    Returns
    -------
    Path
        Path to the generated MP3 file.
    """
    input_path = Path(input_flac_path)
    if not input_path.exists():
        raise FileNotFoundError(f"Source audio file not found: {input_path}")

    if output_mp3_path is None:
        output_mp3_path = WORKSPACE_DIR / f"{input_path.stem}_{bitrate}.mp3"
    else:
        output_mp3_path = Path(output_mp3_path)

    return convert_audio_ffmpeg(input_path, output_mp3_path, bitrate=bitrate, codec="libmp3lame")


def convert_flac_to_multiple_bitrates(
    input_flac_path: str | Path,
    bitrates: list[str] = ("320k", "192k", "128k", "64k", "32k"),
    extra_formats: dict[str, dict] | None = None,
    output_dir: str | Path = WORKSPACE_DIR,
) -> dict[str, dict]:
    """
    Convert a source FLAC file into multiple MP3 bitrates (and optional extra formats)
    and collect compression statistics.
    """
    input_path = Path(input_flac_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Export an uncompressed PCM WAV reference to compare true uncompressed size vs. lossless FLAC vs. lossy MP3
    wav_path = output_dir / f"{input_path.stem}_uncompressed.wav"
    y_orig, sr_orig = sf.read(str(input_path))
    sf.write(str(wav_path), y_orig, sr_orig, format="WAV", subtype="PCM_16")
    wav_size_kb = wav_path.stat().st_size / 1024.0
    flac_size_kb = input_path.stat().st_size / 1024.0

    results = {
        "Uncompressed WAV (PCM)": {
            "path": wav_path,
            "format": "WAV (Uncompressed)",
            "bitrate": "1411k (PCM)",
            "size_kb": wav_size_kb,
            "ratio_vs_flac": flac_size_kb / wav_size_kb,
            "savings_vs_flac_pct": (1.0 - wav_size_kb / flac_size_kb) * 100.0,
        },
        "Original FLAC (Lossless)": {
            "path": input_path,
            "format": "FLAC (Lossless)",
            "bitrate": "Lossless VBR",
            "size_kb": flac_size_kb,
            "ratio_vs_flac": 1.0,
            "savings_vs_flac_pct": 0.0,
        },
    }

    for br in bitrates:
        mp3_out = output_dir / f"{input_path.stem}_{br}.mp3"
        convert_flac_to_mp3(input_path, mp3_out, bitrate=br)
        size_kb = mp3_out.stat().st_size / 1024.0
        results[f"MP3 @ {br}"] = {
            "path": mp3_out,
            "format": "MP3 (Lossy)",
            "bitrate": br,
            "size_kb": size_kb,
            "ratio_vs_flac": flac_size_kb / max(size_kb, 1e-6),
            "savings_vs_flac_pct": (1.0 - size_kb / flac_size_kb) * 100.0,
        }

    if extra_formats:
        for label, cfg in extra_formats.items():
            fmt = cfg["format"]
            br = cfg.get("bitrate", "128k")
            codec = cfg.get("codec", None)
            ext_out = output_dir / f"{input_path.stem}_{fmt}_{br}.{fmt}"
            convert_audio_ffmpeg(input_path, ext_out, bitrate=br, codec=codec)
            size_kb = ext_out.stat().st_size / 1024.0
            results[label] = {
                "path": ext_out,
                "format": fmt.upper(),
                "bitrate": br,
                "size_kb": size_kb,
                "ratio_vs_flac": flac_size_kb / max(size_kb, 1e-6),
                "savings_vs_flac_pct": (1.0 - size_kb / flac_size_kb) * 100.0,
            }

    return results


# Run conversion on the active FLAC file
TARGET_BITRATES = ["320k", "192k", "128k", "64k", "32k"]
EXTRA_FORMATS = {"OGG Vorbis @ 128k": {"format": "ogg", "bitrate": "128k", "codec": "libvorbis"}}

conversion_results = convert_flac_to_multiple_bitrates(
    ACTIVE_FLAC_PATH,
    bitrates=TARGET_BITRATES,
    extra_formats=EXTRA_FORMATS,
)

# Print formatted summary table
print(f"{'Variant':<26} | {'Format':<18} | {'Size (KB)':>10} | {'Ratio vs FLAC':>14} | {'Space Saved vs FLAC':>20}")
print("-" * 98)
for name, info in conversion_results.items():
    print(
        f"{name:<26} | {info['format']:<18} | {info['size_kb']:>10.1f} | "
        f"{info['ratio_vs_flac']:>13.2f}x | {info['savings_vs_flac_pct']:>19.1f}%"
    )

### Interactive Audio Playback

Listen to the uncompressed/lossless audio alongside each compressed MP3 bitrate directly in the notebook:

In [ ]:
for label, info in conversion_results.items():
    if label == "Uncompressed WAV (PCM)":
        continue  # Identical audio stream to Original FLAC
    print(f"{label} ({info['size_kb']:.1f} KB — {info['savings_vs_flac_pct']:.1f}% smaller than FLAC):")
    ipd.display(ipd.Audio(filename=str(info["path"])))


## 4. Spectrogram Comparison: Without Compression (FLAC) vs. With Compression (MP3 Bitrates)

Why use a **Linear-Frequency STFT Spectrogram** first?
- Lossy MP3 encoders apply a **low-pass brickwall filter** whose cutoff frequency drops as the target bitrate decreases:
  - **FLAC / Uncompressed WAV**: Preserves full bandwidth up to the Nyquist limit ($f_s / 2 = 22.05\text{ kHz}$ at $44.1\text{ kHz}$ sample rate).
  - **MP3 @ 320 kbps**: Typical cutoff near $\sim 20\text{ kHz}$.
  - **MP3 @ 192 kbps**: Typical cutoff near $\sim 18.5\text{–}19.5\text{ kHz}$.
  - **MP3 @ 128 kbps**: Typical cutoff near $\sim 16\text{ kHz}$.
  - **MP3 @ 64 kbps**: Typical cutoff near $\sim 11\text{ kHz}$.
  - **MP3 @ 32 kbps**: Typical cutoff near $\sim 7.5\text{–}8\text{ kHz}$.
- On a **linear frequency scale ($0\text{–}22.05\text{ kHz}$)**, the top half of the spectrogram represents $11\text{–}22\text{ kHz}$, making high-frequency truncation and psychoacoustic quantization holes immediately visible.

In [ ]:
def load_and_align_signals(
    conversion_dict: dict[str, dict],
    ref_key: str = "Original FLAC (Lossless)",
    max_duration_sec: float = 10.0,
) -> tuple[int, dict[str, np.ndarray]]:
    """
    Loads the reference FLAC and all compressed audio files at the native sample rate of the FLAC.
    Aligns each compressed signal to the reference using cross-correlation to compensate for MP3 codec encoder delay.
    """
    ref_path = conversion_dict[ref_key]["path"]
    y_ref, sr = librosa.load(str(ref_path), sr=None, mono=True, duration=max_duration_sec)
    signals = {ref_key: y_ref}

    # Use a short slice for fast cross-correlation delay compensation
    align_len = min(len(y_ref), sr * 2)
    for label, info in conversion_dict.items():
        if label == ref_key:
            continue
        y_comp, _ = librosa.load(str(info["path"]), sr=sr, mono=True, duration=max_duration_sec + 0.2)
        # Estimate encoder/decoder frame lag via cross-correlation
        corr = signal.correlate(y_comp[:align_len + 4096], y_ref[:align_len], mode="valid")
        lag = int(np.argmax(np.abs(corr)))
        y_aligned = y_comp[lag : lag + len(y_ref)]
        if len(y_aligned) < len(y_ref):
            y_aligned = np.pad(y_aligned, (0, len(y_ref) - len(y_aligned)))
        signals[label] = y_aligned

    return sr, signals


sr, aligned_signals = load_and_align_signals(conversion_results)

# Select the variants to display in the spectrogram comparison grid
plot_keys = [
    "Original FLAC (Lossless)",
    "MP3 @ 320k",
    "MP3 @ 192k",
    "MP3 @ 128k",
    "MP3 @ 64k",
    "MP3 @ 32k",
]

n_fft = 2048
hop_length = 512

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True, sharey=True)
axes = axes.flatten()

for idx, key in enumerate(plot_keys):
    ax = axes[idx]
    y = aligned_signals[key]
    S_mag = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length))
    S_db = librosa.amplitude_to_db(S_mag, ref=np.max)

    img = librosa.display.specshow(
        S_db,
        sr=sr,
        hop_length=hop_length,
        x_axis="time",
        y_axis="linear",
        cmap="magma",
        vmin=-85,
        vmax=0,
        ax=ax,
    )
    size_kb = conversion_results[key]["size_kb"]
    ax.set_title(f"{key}\n({size_kb:.1f} KB)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)" if idx % 3 == 0 else "")

    # Reference guide lines at 11 kHz, 16 kHz, and 20 kHz
    for f_guide in [11000, 16000, 20000]:
        if f_guide < sr / 2:
            ax.axhline(f_guide, color="cyan", linestyle="--", linewidth=0.8, alpha=0.65)

fig.suptitle(
    "Full-Bandwidth Linear STFT Spectrograms: Lossless FLAC vs. MP3 Bitrates\n"
    "(Cyan dashed lines mark 11 kHz, 16 kHz, and 20 kHz frequency thresholds)",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.72])
fig.colorbar(img, cax=cbar_ax, format="%+2.0f dB", label="Amplitude (dB)")
plt.tight_layout(rect=[0, 0, 0.91, 0.94])
plt.show()

## 5. Residual (Difference) Spectrograms & Mel-Spectrogram Comparison

To isolate **what MP3 compression removes**, we compute the magnitude difference in the STFT domain between the lossless FLAC reference and each compressed version:
$$\Delta S(t, f) = \left| |\text{STFT}_{\text{FLAC}}(t, f)| - |\text{STFT}_{\text{Compressed}}(t, f)| \right|$$
Bright regions in the residual spectrogram correspond to **discarded high frequencies** and **psychoacoustically masked time-frequency bins**.

In [ ]:
y_ref = aligned_signals["Original FLAC (Lossless)"]
S_ref_mag = np.abs(librosa.stft(y_ref, n_fft=n_fft, hop_length=hop_length))
ref_max = np.max(S_ref_mag)

compressed_keys = ["MP3 @ 320k", "MP3 @ 192k", "MP3 @ 128k", "MP3 @ 64k", "MP3 @ 32k", "OGG Vorbis @ 128k"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True, sharey=True)
axes = axes.flatten()

for idx, key in enumerate(compressed_keys):
    ax = axes[idx]
    S_comp_mag = np.abs(librosa.stft(aligned_signals[key], n_fft=n_fft, hop_length=hop_length))
    diff_mag = np.abs(S_ref_mag - S_comp_mag)
    diff_db = librosa.amplitude_to_db(diff_mag, ref=ref_max)

    img = librosa.display.specshow(
        diff_db,
        sr=sr,
        hop_length=hop_length,
        x_axis="time",
        y_axis="linear",
        cmap="inferno",
        vmin=-85,
        vmax=-10,
        ax=ax,
    )
    ax.set_title(f"Lost Signal / Residual: FLAC vs. {key}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)" if idx % 3 == 0 else "")

fig.suptitle(
    "Residual (Difference) Spectrograms |STFT_FLAC - STFT_Compressed| (Brighter = More Information Lost)",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.72])
fig.colorbar(img, cax=cbar_ax, format="%+2.0f dB", label="Residual Error Energy (dB rel. to peak)")
plt.tight_layout(rect=[0, 0, 0.91, 0.94])
plt.show()

### Perceptual Mel-Spectrogram Comparison

Below is the **Mel-scale spectrogram** comparison (128 Mel bands up to $f_{\max} = f_s / 2$), which spaces frequency bins logarithmically according to human pitch perception:

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9), sharex=True, sharey=True)
axes = axes.flatten()

for idx, key in enumerate(plot_keys):
    ax = axes[idx]
    S_mel = librosa.feature.melspectrogram(
        y=aligned_signals[key],
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=128,
        fmax=sr / 2,
    )
    S_mel_db = librosa.power_to_db(S_mel, ref=np.max)

    img = librosa.display.specshow(
        S_mel_db,
        sr=sr,
        hop_length=hop_length,
        x_axis="time",
        y_axis="mel",
        fmax=sr / 2,
        cmap="viridis",
        vmin=-80,
        vmax=0,
        ax=ax,
    )
    ax.set_title(f"Mel Spectrogram: {key}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz - Mel Scale)" if idx % 3 == 0 else "")

fig.suptitle("Perceptual Mel-Spectrogram (128 Mel Bands): FLAC vs. MP3 Bitrates", fontsize=14, fontweight="bold", y=0.98)
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.72])
fig.colorbar(img, cax=cbar_ax, format="%+2.0f dB", label="Power (dB)")
plt.tight_layout(rect=[0, 0, 0.91, 0.94])
plt.show()

## 6. Power Spectral Density (PSD) Roll-Off & Quantitative Quality Metrics

By averaging the STFT power across time for each frequency bin, we obtain the **Mean Power Spectral Density (PSD)** profile across $0\text{–}22.05\text{ kHz}$. This single plot clearly reveals the exact high-frequency low-pass filter knee for each MP3 bitrate.

We also compute:
- **Log-Spectral Distance (LSD)** in dB (lower means closer to lossless FLAC).
- **Effective Bandwidth Cutoff ($f_{-65\text{ dB}}$)** in kHz (the highest frequency retaining energy within 65 dB of the spectral peak).

In [ ]:
freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft) / 1000.0  # kHz

palette = {
    "Original FLAC (Lossless)": "#111111",
    "MP3 @ 320k": "#1f77b4",
    "MP3 @ 192k": "#2ca02c",
    "MP3 @ 128k": "#ff7f0e",
    "MP3 @ 64k": "#d62728",
    "MP3 @ 32k": "#9467bd",
    "OGG Vorbis @ 128k": "#17becf",
}

fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 2, height_ratios=[1.15, 1.0], hspace=0.35, wspace=0.25)

# 1. Top Panel: Average Power Spectral Density (PSD) vs. Frequency (kHz)
ax_psd = fig.add_subplot(gs[0, :])

S_ref_power = np.abs(librosa.stft(aligned_signals["Original FLAC (Lossless)"], n_fft=n_fft, hop_length=hop_length)) ** 2
ref_peak_power = np.max(np.mean(S_ref_power, axis=1))
S_ref_log = 10.0 * np.log10(np.maximum(S_ref_power / ref_peak_power, 1e-10))

metrics_summary = []

for key, color in palette.items():
    if key not in aligned_signals:
        continue
    y = aligned_signals[key]
    S_pow = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length)) ** 2
    mean_psd_db = 10.0 * np.log10(np.maximum(np.mean(S_pow, axis=1) / ref_peak_power, 1e-10))

    lw = 2.4 if "FLAC" in key else 1.7
    ls = "-" if "OGG" not in key else "--"
    ax_psd.plot(freqs, mean_psd_db, label=key, color=color, linewidth=lw, linestyle=ls, alpha=0.9)

    # Compute Log-Spectral Distance (LSD) relative to Original FLAC
    S_log = 10.0 * np.log10(np.maximum(S_pow / ref_peak_power, 1e-10))
    lsd_db = float(np.mean(np.sqrt(np.mean((S_ref_log - S_log) ** 2, axis=0))))

    # Estimate effective high-frequency cutoff (-65 dB threshold relative to peak)
    active_bins = np.where(mean_psd_db > -65.0)[0]
    cutoff_khz = float(freqs[active_bins[-1]]) if len(active_bins) > 0 else 0.0

    metrics_summary.append({
        "label": key,
        "size_kb": conversion_results[key]["size_kb"],
        "lsd_db": lsd_db,
        "cutoff_khz": cutoff_khz,
        "color": color,
    })

ax_psd.set_title("Mean Power Spectral Density (PSD) Across Frequency: Visualizing MP3 Brickwall Cutoffs", fontsize=13, fontweight="bold")
ax_psd.set_xlabel("Frequency (kHz)", fontsize=11)
ax_psd.set_ylabel("Mean Power (dB rel. to peak)", fontsize=11)
ax_psd.set_xlim(0, (sr / 2) / 1000.0)
ax_psd.set_ylim(-95, 5)
ax_psd.grid(True, linestyle=":", alpha=0.5)
ax_psd.legend(loc="upper right", ncol=2, frameon=True)

# 2. Bottom-Left Panel: File Size Comparison (KB)
ax_size = fig.add_subplot(gs[1, 0])
labels_short = [m["label"].replace("Original ", "").replace(" (Lossless)", "") for m in metrics_summary]
sizes = [m["size_kb"] for m in metrics_summary]
colors = [m["color"] for m in metrics_summary]

bars1 = ax_size.bar(labels_short, sizes, color=colors, alpha=0.88)
ax_size.set_title("File Size Comparison (KB — Lower is Smaller)", fontsize=12, fontweight="bold")
ax_size.set_ylabel("File Size (KB)")
ax_size.tick_params(axis="x", rotation=25)
for bar in bars1:
    h = bar.get_height()
    ax_size.annotate(f"{h:.0f} KB", xy=(bar.get_x() + bar.get_width() / 2, h), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)

# 3. Bottom-Right Panel: Log-Spectral Distance (LSD) & Effective Bandwidth
ax_lsd = fig.add_subplot(gs[1, 1])
lsd_vals = [m["lsd_db"] for m in metrics_summary]
bars2 = ax_lsd.bar(labels_short, lsd_vals, color=colors, alpha=0.88)
ax_lsd.set_title("Spectral Distortion: Log-Spectral Distance vs. FLAC (dB — Lower is Better)", fontsize=12, fontweight="bold")
ax_lsd.set_ylabel("Log-Spectral Distance (dB)")
ax_lsd.tick_params(axis="x", rotation=25)
for bar, m in zip(bars2, metrics_summary):
    h = bar.get_height()
    ax_lsd.annotate(
        f"{h:.2f} dB\n({m['cutoff_khz']:.1f} kHz)",
        xy=(bar.get_x() + bar.get_width() / 2, h),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        fontsize=8.5,
    )

plt.show()

### Q&A
- **How can I compare spectrograms of uncompressed/lossless audio (FLAC) against compressed MP3 bitrates?**
  Use the linear-frequency STFT spectrogram grid (Section 4) and residual difference spectrogram grid (Section 5) to inspect both full-spectrum preservation up to the Nyquist frequency ($f_s / 2$) and the exact time-frequency bins removed by MP3 psychoacoustic encoding at `320k`, `192k`, `128k`, `64k`, and `32k`.
- **How do I upload a local FLAC or audio file and convert it to MP3?**
  Use the file upload widget or local file path loader in Section 2, then run `convert_flac_to_mp3()` or `convert_flac_to_multiple_bitrates()` in Section 3.

### Data Analysis Key Findings
- **High-Frequency Brickwall Cutoffs**: Lower MP3 bitrates enforce progressively stricter low-pass filters (~20 kHz at `320k`, ~19 kHz at `192k`, ~16 kHz at `128k`, ~11 kHz at `64k`, and ~8 kHz at `32k`), visible as dark horizontal bands at the top of the linear STFT spectrograms and steep cliffs in the Power Spectral Density curve.
- **Psychoacoustic Masking Artifacts**: Even below the low-pass cutoff frequency, the residual difference spectrograms (`|STFT_FLAC - STFT_MP3|`) reveal quantization noise and time-frequency holes where quieter harmonics are masked by louder nearby frequencies.
- **Compression Ratio vs. Spectral Fidelity**: `192k` and `320k` MP3 achieve substantial file size reductions relative to lossless FLAC while maintaining low Log-Spectral Distance (LSD), whereas bitrates below `128k` exhibit steep increases in spectral distortion.

### Insights or Next Steps
- **Evaluate Neural Audio Codecs**: Extend `EXTRA_FORMATS` or add neural audio compression models (such as EnCodec or Descript Audio Codec) to compare neural spectral reconstruction against traditional MP3/Vorbis psychoacoustic codecs.
- **Bandwidth Extension / Audio Super-Resolution**: Use the paired `(MP3 @ 64k/128k, Original FLAC)` spectrograms generated in `audio_workspace/` as training pairs for a spectral enhancement or bandwidth-extension neural network.